# TrueRender v2 — Upgraded Pipeline

| Stage | Tool | Status |
|---|---|---|
| 1. Frame extract + filter | ffmpeg + Laplacian | ✅ implemented |
| 2. Object detection | Grounding DINO | 🔜 next |
| 3. Masking | SAM 3 | 🔜 |
| 4. Geometry | VGGT | 🔜 |
| 5. Splat training | 2DGS | 🔜 |
| 6. Mesh extract | TSDF | 🔜 |
| 7. Export | trimesh GLB/STL/OBJ | 🔜 |



## Setup
Run once per Colab session.

In [ ]:
!pip install -q opencv-python-headless matplotlib
print("Done")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Stage 1 — Frame Extraction + Quality Filter

Extract raw frames at 5 fps, then keep the **30 sharpest** by variance-of-Laplacian.
Blurry frames hurt geometry estimation — this filter is free and cuts 57% of frames,
directly reducing compute in every downstream stage.

In [ ]:
import cv2, glob, shutil, os, time

VIDEO_PATH  = "/content/drive/MyDrive/hydroflaskgreen.MOV"
FRAMES_RAW  = "/content/frames_raw"
FRAMES_KEEP = "/content/frames"
N_KEEP      = 30

os.makedirs(FRAMES_RAW,  exist_ok=True)
os.makedirs(FRAMES_KEEP, exist_ok=True)

t0 = time.time()

# Extract raw frames
!ffmpeg -i "{VIDEO_PATH}" -vf fps=5 "{FRAMES_RAW}/frame_%05d.jpg" -y -loglevel error

all_frames = sorted(glob.glob(f"{FRAMES_RAW}/*.jpg"))
print(f"Raw frames extracted: {len(all_frames)}")

# Score by variance-of-Laplacian (higher = sharper)
scored = [
    (f, cv2.Laplacian(cv2.imread(f, cv2.IMREAD_GRAYSCALE), cv2.CV_64F).var())
    for f in all_frames
]
scored.sort(key=lambda x: -x[1])

# Keep N_KEEP sharpest, re-sort by filename to preserve temporal order
keep = sorted([f for f, _ in scored[:N_KEEP]])
for dst_idx, src in enumerate(keep):
    shutil.copy(src, f"{FRAMES_KEEP}/frame_{dst_idx:05d}.jpg")

t1 = time.time()
print(f"Kept {len(keep)} / {len(all_frames)} frames")
print(f"Sharpness — best: {scored[0][1]:.1f}  cutoff: {scored[N_KEEP-1][1]:.1f}  worst dropped: {scored[-1][1]:.1f}")
print(f"Stage 1 wall time: {t1 - t0:.1f}s")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

frames = sorted(os.listdir(FRAMES_KEEP))
indices = [0, len(frames)//3, 2*len(frames)//3, -1]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, i in zip(axes, indices):
    ax.imshow(mpimg.imread(f"{FRAMES_KEEP}/{frames[i]}"))
    ax.set_title(frames[i])
    ax.axis("off")
plt.suptitle(f"Top-{N_KEEP} sharpest frames (Laplacian filter)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Checkpoint to Drive — so a fresh session can skip re-extraction
DRIVE_FRAMES = "/content/drive/MyDrive/truerender_frames_v2"
if os.path.exists(DRIVE_FRAMES):
    shutil.rmtree(DRIVE_FRAMES)
shutil.copytree(FRAMES_KEEP, DRIVE_FRAMES)
print(f"Checkpointed {len(os.listdir(DRIVE_FRAMES))} frames → Drive")